In [1]:
# ============================================================
# 1. MOUNT GOOGLE DRIVE AND INSTALL DEPENDENCIES
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

!pip -q install transformers accelerate pandas pillow tqdm

Mounted at /content/drive


In [4]:
# ============================================================
# 2. IMPORT LIBRARIES AND CONFIGURATION
# ============================================================

import os
import random
import pandas as pd
import numpy as np
import torch

from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration
)

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

# ------------------------------------------------------------
# GOOGLE DRIVE PATHS
# ------------------------------------------------------------

PROJECT_DIR = (
    "/content/drive/MyDrive/EEE6778_Semester_Project"
)

IMAGE_DIR = os.path.join(
    PROJECT_DIR,
    "doclaynet_synthetic_dataset",
    "images"
)

OUTPUT_CSV = os.path.join(
    PROJECT_DIR,
    "BLIP_generated_dataset.csv"
)

CAPTIONS_CSV = os.path.join(
    PROJECT_DIR,
    "BLIP_image_captions.csv"
)

os.makedirs(PROJECT_DIR, exist_ok=True)

# ------------------------------------------------------------
# BLIP CONFIGURATION
# ------------------------------------------------------------

BLIP_MODEL_NAME = "Salesforce/blip-image-captioning-base"

MAX_NEW_TOKENS = 75
NUM_BEAMS = 5

# Number of mismatched captions per image.
NEGATIVES_PER_IMAGE = 1

# Prefer negative captions from the same document.
PREFER_SAME_DOCUMENT = True

print("Image directory:", IMAGE_DIR)
print("Output dataset:", OUTPUT_CSV)

Device: cpu
Image directory: /content/drive/MyDrive/EEE6778_Semester_Project/doclaynet_synthetic_dataset/images
Output dataset: /content/drive/MyDrive/EEE6778_Semester_Project/BLIP_generated_dataset.csv


In [5]:
# ============================================================
# 3. LOAD EXTRACTED IMAGE DATASET
# ============================================================

# OPTIONAL:
# Enter the path to your existing image metadata CSV.
# Expected columns:
# document_id, page_id, figure_id, image_path
#
# Leave as None to discover images automatically.

METADATA_CSV = None

VALID_EXTENSIONS = {
    ".png",
    ".jpg",
    ".jpeg",
    ".webp",
    ".bmp",
    ".tif",
    ".tiff"
}


def find_column(df, candidates):

    lookup = {
        str(col).lower(): col
        for col in df.columns
    }

    for candidate in candidates:

        if candidate.lower() in lookup:
            return lookup[candidate.lower()]

    return None


# ------------------------------------------------------------
# LOAD FROM METADATA CSV IF PROVIDED
# ------------------------------------------------------------

if METADATA_CSV is not None:

    source_df = pd.read_csv(METADATA_CSV)

    image_col = find_column(
        source_df,
        ["image_path", "image", "filepath", "file_path"]
    )

    if image_col is None:
        raise ValueError(
            "Metadata CSV must contain an image_path column."
        )

    records = []

    for _, row in source_df.iterrows():

        image_path = str(row[image_col])

        if not os.path.isabs(image_path):

            image_path = os.path.join(
                IMAGE_DIR,
                image_path
            )

        if not os.path.isfile(image_path):
            continue

        records.append({

            "document_id": row.get("document_id", None),

            "page_id": row.get("page_id", None),

            "figure_id": row.get("figure_id", None),

            "image_path": image_path

        })

    images_df = pd.DataFrame(records)


# ------------------------------------------------------------
# OTHERWISE, DISCOVER IMAGES AUTOMATICALLY
# ------------------------------------------------------------

else:

    if not os.path.isdir(IMAGE_DIR):

        raise FileNotFoundError(
            f"Image directory not found: {IMAGE_DIR}"
        )

    image_paths = sorted(

        str(path)

        for path in Path(IMAGE_DIR).rglob("*")

        if (
            path.is_file()
            and path.suffix.lower() in VALID_EXTENSIONS
        )

    )

    images_df = pd.DataFrame({

        "image_path": image_paths

    })

    # Use parent directory as a fallback document grouping.
    images_df["document_id"] = (
        images_df["image_path"]
        .apply(lambda p: Path(p).parent.name)
    )

    images_df["page_id"] = None

    images_df["figure_id"] = (
        images_df["image_path"]
        .apply(lambda p: Path(p).stem)
    )


# ------------------------------------------------------------
# CLEAN DATASET
# ------------------------------------------------------------

images_df = images_df.drop_duplicates(
    subset=["image_path"]
).reset_index(drop=True)

if len(images_df) < 2:

    raise ValueError(
        "At least two images are required to generate "
        "matched and mismatched pairs."
    )


print("Total extracted images:", len(images_df))

display(images_df.head())

Total extracted images: 1001


,image_path,document_id,page_id,figure_id
0,/content/drive/MyDrive/EEE6778_Semester_Projec...,images,None,doc_1001_figure_0
1,/content/drive/MyDrive/EEE6778_Semester_Projec...,images,None,doc_1002_figure_0
2,/content/drive/MyDrive/EEE6778_Semester_Projec...,images,None,doc_1002_figure_1
3,/content/drive/MyDrive/EEE6778_Semester_Projec...,images,None,doc_1002_figure_2
4,/content/drive/MyDrive/EEE6778_Semester_Projec...,images,None,doc_1004_figure_0


In [6]:
# ============================================================
# 4. LOAD BLIP MODEL
# ============================================================

processor = BlipProcessor.from_pretrained(
    BLIP_MODEL_NAME
)

model = BlipForConditionalGeneration.from_pretrained(
    BLIP_MODEL_NAME
).to(DEVICE)

model.eval()

print("BLIP model loaded successfully.")

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

BLIP model loaded successfully.


In [7]:
# ============================================================
# 5. GENERATE BLIP CAPTIONS
# ============================================================

def generate_blip_caption(image_path):

    try:

        with Image.open(image_path) as image:

            image = image.convert("RGB")

            inputs = processor(
                images=image,
                return_tensors="pt"
            ).to(DEVICE)

            with torch.inference_mode():

                output = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    num_beams=NUM_BEAMS
                )

            caption = processor.decode(
                output[0],
                skip_special_tokens=True
            )

        return caption.strip()

    except Exception as e:

        print(
            f"Error processing {image_path}: {e}"
        )

        return None


# ------------------------------------------------------------
# GENERATE CAPTIONS FOR ALL IMAGES
# ------------------------------------------------------------

captions = []

for image_path in tqdm(
    images_df["image_path"],
    desc="Generating BLIP captions"
):

    caption = generate_blip_caption(image_path)

    captions.append(caption)


images_df["blip_caption"] = captions


# ------------------------------------------------------------
# REMOVE FAILED CAPTIONS
# ------------------------------------------------------------

images_df = images_df.dropna(
    subset=["blip_caption"]
).copy()

images_df["blip_caption"] = (
    images_df["blip_caption"]
    .astype(str)
    .str.strip()
)

images_df = images_df[
    images_df["blip_caption"] != ""
].reset_index(drop=True)


if len(images_df) < 2:

    raise ValueError(
        "Fewer than two valid BLIP captions were generated."
    )


# ------------------------------------------------------------
# SAVE CAPTION LOOKUP
# ------------------------------------------------------------

images_df.to_csv(
    CAPTIONS_CSV,
    index=False
)

print("Successfully generated captions:", len(images_df))

print("Caption file saved:", CAPTIONS_CSV)

display(
    images_df[
        ["image_path", "blip_caption"]
    ].head(10)
)

Generating BLIP captions:   0%|          | 0/1001 [00:00<?, ?it/s]

Successfully generated captions: 1001
Caption file saved: /content/drive/MyDrive/EEE6778_Semester_Project/BLIP_image_captions.csv


,image_path,blip_caption
0,/content/drive/MyDrive/EEE6778_Semester_Projec...,the key bank logo
1,/content/drive/MyDrive/EEE6778_Semester_Projec...,two men in suits standing next to each other men
2,/content/drive/MyDrive/EEE6778_Semester_Projec...,a group of three men standing next to each oth...
3,/content/drive/MyDrive/EEE6778_Semester_Projec...,a group of people standing next to each other ...
4,/content/drive/MyDrive/EEE6778_Semester_Projec...,a man in a blue shirt and tie smiles at the ca...
5,/content/drive/MyDrive/EEE6778_Semester_Projec...,a bar chart showing the percentage of the numb...
6,/content/drive/MyDrive/EEE6778_Semester_Projec...,a bar chart showing the average cost of a home
7,/content/drive/MyDrive/EEE6778_Semester_Projec...,a black and white drawing of a fish
8,/content/drive/MyDrive/EEE6778_Semester_Projec...,a man in a suit and tie posing for a picture
9,/content/drive/MyDrive/EEE6778_Semester_Projec...,an image of a yellow background with a black a...


In [8]:
# ============================================================
# 6. GENERATE POSITIVE AND NEGATIVE PAIRS
# ============================================================

images_df = images_df.reset_index(drop=True)

# Assign a unique identifier to every source image.
images_df["source_image_id"] = (
    images_df.index.astype(str)
)

pair_records = []


for idx, row in tqdm(
    images_df.iterrows(),
    total=len(images_df),
    desc="Creating image-caption pairs"
):

    target_image = row["image_path"]

    target_caption = row["blip_caption"]

    target_document = row["document_id"]


    # ========================================================
    # POSITIVE PAIR
    # ========================================================

    # Image is paired with its own BLIP caption.

    pair_records.append({

        "document_id": row["document_id"],

        "page_id": row["page_id"],

        "figure_id": row["figure_id"],

        "image_path": target_image,

        "alt_text": target_caption,

        "label": 1,

        "pair_type": "positive",

        "caption_source_image_path": target_image,

        "caption_source_image_id": row["source_image_id"]

    })


    # ========================================================
    # NEGATIVE PAIR
    # ========================================================

    # Find captions generated from other images.

    candidates = images_df[
        images_df.index != idx
    ]


    # Prefer captions from another image in the same document.

    if (
        PREFER_SAME_DOCUMENT
        and pd.notna(target_document)
    ):

        same_document_candidates = candidates[

            candidates["document_id"].astype(str)
            == str(target_document)

        ]

        # If other images exist in the same document,
        # use them as negative examples.

        if len(same_document_candidates) > 0:

            candidates = same_document_candidates


    if len(candidates) == 0:
        continue


    # Select negative captions.

    n_to_sample = min(
        NEGATIVES_PER_IMAGE,
        len(candidates)
    )

    sampled_candidates = candidates.sample(
        n=n_to_sample,
        random_state=SEED + idx
    )


    for _, negative_row in sampled_candidates.iterrows():

        pair_records.append({

            # Metadata belongs to the target image.

            "document_id": row["document_id"],

            "page_id": row["page_id"],

            "figure_id": row["figure_id"],

            # Target image.

            "image_path": target_image,

            # Caption comes from a DIFFERENT image.

            "alt_text": negative_row["blip_caption"],

            # Negative label.

            "label": 0,

            "pair_type": "negative",

            # Track the original image used to generate
            # the negative caption.

            "caption_source_image_path": negative_row["image_path"],

            "caption_source_image_id": negative_row["source_image_id"]

        })


# ============================================================
# COMBINE DATASET
# ============================================================

dataset_df = pd.DataFrame(pair_records)


# Shuffle the dataset.

dataset_df = dataset_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)


# ============================================================
# VALIDATE NEGATIVE PAIRS
# ============================================================

negative_df = dataset_df[
    dataset_df["label"] == 0
]

assert (
    negative_df["image_path"]
    != negative_df["caption_source_image_path"]
).all(), "Invalid negative pair detected!"


print("Total image-caption pairs:", len(dataset_df))

print("\nPair distribution:")

print(
    dataset_df["pair_type"].value_counts()
)

print("\nLabel distribution:")

print(
    dataset_df["label"].value_counts()
)

display(dataset_df.head(10))

Creating image-caption pairs:   0%|          | 0/1001 [00:00<?, ?it/s]

Total image-caption pairs: 2002

Pair distribution:
pair_type
positive    1001
negative    1001
Name: count, dtype: int64

Label distribution:
label
1    1001
0    1001
Name: count, dtype: int64


,document_id,page_id,figure_id,image_path,alt_text,label,pair_type,caption_source_image_path,caption_source_image_id
0,images,None,doc_514_figure_0,/content/drive/MyDrive/EEE6778_Semester_Projec...,an aerial view of a bridge over a body of water,1,positive,/content/drive/MyDrive/EEE6778_Semester_Projec...,697
1,images,None,doc_1221_figure_8,/content/drive/MyDrive/EEE6778_Semester_Projec...,a black and white photo of a smiling man,0,negative,/content/drive/MyDrive/EEE6778_Semester_Projec...,485
2,images,None,doc_459_figure_6,/content/drive/MyDrive/EEE6778_Semester_Projec...,the signature of a man in blue ink on a white ...,1,positive,/content/drive/MyDrive/EEE6778_Semester_Projec...,667
3,images,None,doc_1695_figure_0,/content/drive/MyDrive/EEE6778_Semester_Projec...,"a table with two columns, one column and one c...",1,positive,/content/drive/MyDrive/EEE6778_Semester_Projec...,453
4,images,None,doc_420_figure_1,/content/drive/MyDrive/EEE6778_Semester_Projec...,a pie pie pie pie pie pie pie pie pie pie pie ...,1,positive,/content/drive/MyDrive/EEE6778_Semester_Projec...,645
5,images,None,doc_410_figure_2,/content/drive/MyDrive/EEE6778_Semester_Projec...,a group of people walking around a building,1,positive,/content/drive/MyDrive/EEE6778_Semester_Projec...,637
6,images,None,doc_1707_figure_0,/content/drive/MyDrive/EEE6778_Semester_Projec...,a blue and white logo with the letter d and a ...,0,negative,/content/drive/MyDrive/EEE6778_Semester_Projec...,620
7,images,None,doc_763_figure_0,/content/drive/MyDrive/EEE6778_Semester_Projec...,the british flag on a white background,1,positive,/content/drive/MyDrive/EEE6778_Semester_Projec...,866
8,images,None,doc_1018_figure_0,/content/drive/MyDrive/EEE6778_Semester_Projec...,a group of logos for various brands,0,negative,/content/drive/MyDrive/EEE6778_Semester_Projec...,998
9,images,None,doc_200_figure_9,/content/drive/MyDrive/EEE6778_Semester_Projec...,a picture of a machine with a blue background ...,0,negative,/content/drive/MyDrive/EEE6778_Semester_Projec...,679


In [9]:
# ============================================================
# 7. SAVE BLIP_GENERATED_DATASET.CSV
# ============================================================

# Organize columns.

output_columns = [

    "document_id",

    "page_id",

    "figure_id",

    "image_path",

    "alt_text",

    "label",

    "pair_type",

    "caption_source_image_path",

    "caption_source_image_id"

]

dataset_df = dataset_df[output_columns]


# Save to Google Drive.

dataset_df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8"
)


# ============================================================
# FINAL DATASET SUMMARY
# ============================================================

positive_count = (
    dataset_df["label"] == 1
).sum()

negative_count = (
    dataset_df["label"] == 0
).sum()

print("=" * 55)

print("BLIP DATASET GENERATION COMPLETE")

print("=" * 55)

print("Output file:", OUTPUT_CSV)

print("Total pairs:", len(dataset_df))

print("Positive pairs:", positive_count)

print("Negative pairs:", negative_count)

print(
    "Balanced dataset:",
    positive_count == negative_count
)

print("=" * 55)


# Verify file exists.

assert os.path.isfile(OUTPUT_CSV)

print("\nFile successfully saved to Google Drive!")

BLIP DATASET GENERATION COMPLETE
Output file: /content/drive/MyDrive/EEE6778_Semester_Project/BLIP_generated_dataset.csv
Total pairs: 2002
Positive pairs: 1001
Negative pairs: 1001
Balanced dataset: True

File successfully saved to Google Drive!


In [10]:
# ============================================================
# 8. INSPECT GENERATED DATASET
# ============================================================

dataset_check = pd.read_csv(OUTPUT_CSV)

print("Dataset shape:", dataset_check.shape)

print("\nLabel counts:")

print(
    dataset_check["label"].value_counts()
)

print("\nExample positive pairs:")

display(
    dataset_check[
        dataset_check["label"] == 1
    ][
        ["image_path", "alt_text", "label"]
    ].head(5)
)


print("\nExample negative pairs:")

display(
    dataset_check[
        dataset_check["label"] == 0
    ][
        [
            "image_path",
            "alt_text",
            "caption_source_image_path",
            "label"
        ]
    ].head(5)
)

Dataset shape: (2002, 9)

Label counts:
label
1    1001
0    1001
Name: count, dtype: int64

Example positive pairs:


,image_path,alt_text,label
0,/content/drive/MyDrive/EEE6778_Semester_Projec...,an aerial view of a bridge over a body of water,1
2,/content/drive/MyDrive/EEE6778_Semester_Projec...,the signature of a man in blue ink on a white ...,1
3,/content/drive/MyDrive/EEE6778_Semester_Projec...,"a table with two columns, one column and one c...",1
4,/content/drive/MyDrive/EEE6778_Semester_Projec...,a pie pie pie pie pie pie pie pie pie pie pie ...,1
5,/content/drive/MyDrive/EEE6778_Semester_Projec...,a group of people walking around a building,1



Example negative pairs:


,image_path,alt_text,caption_source_image_path,label
1,/content/drive/MyDrive/EEE6778_Semester_Projec...,a black and white photo of a smiling man,/content/drive/MyDrive/EEE6778_Semester_Projec...,0
6,/content/drive/MyDrive/EEE6778_Semester_Projec...,a blue and white logo with the letter d and a ...,/content/drive/MyDrive/EEE6778_Semester_Projec...,0
8,/content/drive/MyDrive/EEE6778_Semester_Projec...,a group of logos for various brands,/content/drive/MyDrive/EEE6778_Semester_Projec...,0
9,/content/drive/MyDrive/EEE6778_Semester_Projec...,a picture of a machine with a blue background ...,/content/drive/MyDrive/EEE6778_Semester_Projec...,0
11,/content/drive/MyDrive/EEE6778_Semester_Projec...,a man in a white shirt and a man in a blue tie,/content/drive/MyDrive/EEE6778_Semester_Projec...,0
